<a href="https://colab.research.google.com/github/raymondtyaba/case-manager-dashboard/blob/main/Full_RAG_Studio_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Full RAG Studio — Capstone
**John Raymond TYABA | 23 September 2026**

End-to-end RAG: PDF/OCR → chunks + metadata → MiniLM embeddings → FAISS → grounded prompt → FLAN-T5 → answer, sources, confidence and chunk count.

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr poppler-utils
!pip -q install --upgrade gradio pypdf sentence-transformers faiss-cpu transformers pdf2image pytesseract pillow

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.9_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.9) ...
Setting up poppler-utils (24.02.0-1ubuntu9.9) ...
Processing triggers for man-db (2.12.0-4build2) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 42.9 MB/

In [ ]:
"""Full RAG Studio: OCR + metadata + sentence-transformer embeddings + FAISS + FLAN-T5 + Gradio."""
from __future__ import annotations
import re, time, tempfile
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Optional
import numpy as np
import gradio as gr
from pypdf import PdfReader

@dataclass
class Chunk:
    text: str
    source: str
    source_id: str
    page_start: int
    page_end: int
    doc_type: str
    chunk_id: str

class DocumentProcessor:
    def __init__(self, chunk_size=900, overlap=150):
        self.chunk_size, self.overlap = chunk_size, overlap

    @staticmethod
    def classify(text: str) -> str:
        t=text.lower()
        rules={
            "Certificate of Quality":["certificate of quality","certificate of analysis","batch","assay"],
            "Packaging Specification":["packaging specification","label","carton","blister"],
            "Compliance Declaration":["compliance","declaration","regulation","conformity"],
            "Research Report":["abstract","method","results","conclusion","research"],
        }
        scores={k:sum(term in t for term in terms) for k,terms in rules.items()}
        best=max(scores,key=scores.get)
        return best if scores[best] else "General Document"

    @staticmethod
    def ocr_page(pdf_path: str, page_number: int) -> str:
        """OCR one page. Requires poppler and Tesseract, installed by the notebook."""
        from pdf2image import convert_from_path
        import pytesseract
        images=convert_from_path(pdf_path, first_page=page_number, last_page=page_number, dpi=250)
        return pytesseract.image_to_string(images[0]) if images else ""

    def extract(self, path: str):
        reader=PdfReader(path); pages=[]; ocr_pages=[]
        for i,page in enumerate(reader.pages,1):
            text=(page.extract_text() or "").strip()
            if len(re.sub(r"\s+","",text)) < 40:
                try:
                    text=self.ocr_page(path,i).strip(); ocr_pages.append(i)
                except Exception as exc:
                    text=f"[OCR unavailable for page {i}: {exc}]"
            pages.append((i,re.sub(r"\s+"," ",text).strip()))
        return pages,ocr_pages

    def process(self, paths: list[str]):
        chunks=[]; summaries=[]
        for file_no,path in enumerate(paths,1):
            pages,ocr_pages=self.extract(path); source=Path(path).name; source_id=f"DOC-{file_no:03d}"
            for page_no,text in pages:
                if not text or text.startswith("[OCR unavailable"): continue
                dtype=self.classify(text); start=0; n=0
                while start < len(text):
                    end=min(start+self.chunk_size,len(text)); n+=1
                    chunks.append(Chunk(text[start:end],source,source_id,page_no,page_no,dtype,f"{source_id}-P{page_no}-C{n}"))
                    if end==len(text): break
                    start=max(end-self.overlap,start+1)
            summaries.append({"file":source,"pages":len(pages),"ocr_pages":ocr_pages})
        return chunks,summaries

class RAGEngine:
    def __init__(self, embedding_model="sentence-transformers/all-MiniLM-L6-v2", llm_model="google/flan-t5-small"):
        self.embedding_model_name=embedding_model; self.llm_model_name=llm_model
        self.embedder=None; self.generator=None; self.index=None; self.chunks=[]

    def load_models(self):
        if self.embedder is None:
            from sentence_transformers import SentenceTransformer
            self.embedder=SentenceTransformer(self.embedding_model_name)
        if self.generator is None:
            from transformers import pipeline
            self.generator=pipeline("text2text-generation",model=self.llm_model_name,device=-1)

    def build_index(self,chunks):
        import faiss
        self.load_models(); self.chunks=list(chunks)
        vectors=self.embedder.encode([c.text for c in chunks],normalize_embeddings=True,show_progress_bar=False)
        vectors=np.asarray(vectors,dtype="float32")
        self.index=faiss.IndexFlatIP(vectors.shape[1]); self.index.add(vectors)

    def retrieve(self,query,k=4,doc_type="All",auto_route=True):
        if self.index is None or not self.chunks: return []
        q=np.asarray(self.embedder.encode([query],normalize_embeddings=True),dtype="float32")
        search_k=min(max(k*5,k),len(self.chunks)); scores,ids=self.index.search(q,search_k)
        candidates=[]
        routed=DocumentProcessor.classify(query) if auto_route else "General Document"
        effective_filter=doc_type if doc_type!="All" else (routed if auto_route and routed!="General Document" else None)
        for score,idx in zip(scores[0],ids[0]):
            if idx<0: continue
            c=self.chunks[int(idx)]
            if effective_filter and c.doc_type!=effective_filter: continue
            candidates.append((c,float(score)))
            if len(candidates)>=k: break
        if not candidates and effective_filter:
            return self.retrieve(query,k,"All",False)
        return candidates

    def answer(self,query,k=4,doc_type="All",auto_route=True):
        hits=self.retrieve(query,k,doc_type,auto_route)
        if not hits:
            return {"answer":"I could not find relevant information in the indexed documents.","sources":[],"confidence":0.0,"chunks_used":0}
        context="\n\n".join(f"[{i}] {c.source}, page {c.page_start}, type {c.doc_type}:\n{c.text}" for i,(c,_) in enumerate(hits,1))
        prompt=("Answer the question using only the context. If the context is insufficient, say so. "
                "Give a concise factual answer and cite evidence with [1], [2], etc.\n\n"
                f"CONTEXT:\n{context}\n\nQUESTION: {query}\nANSWER:")
        self.load_models()
        output=self.generator(prompt,max_new_tokens=220,do_sample=False,truncation=True)[0]["generated_text"].strip()
        sources=[{"source":c.source,"page":c.page_start,"document_type":c.doc_type,"chunk_id":c.chunk_id,"score":round(s,3)} for c,s in hits]
        confidence=max(0.0,min(1.0,float(np.mean([max(0,s) for _,s in hits]))))
        return {"answer":output,"sources":sources,"confidence":confidence,"chunks_used":len(hits)}

PROCESSOR=DocumentProcessor(); ENGINE=RAGEngine()

def file_paths(files):
    if not files:return []
    files=files if isinstance(files,list) else [files]
    return [f if isinstance(f,str) else getattr(f,"name",f.get("path") if isinstance(f,dict) else "") for f in files]

def process_ui(files):
    paths=[p for p in file_paths(files) if p]
    if not paths:return "Upload at least one PDF.",[],gr.update(choices=["All"],value="All")
    started=time.perf_counter(); chunks,summary=PROCESSOR.process(paths)
    if not chunks:return "No readable text was extracted.",[],gr.update(choices=["All"],value="All")
    ENGINE.build_index(chunks); elapsed=time.perf_counter()-started
    types=sorted({c.doc_type for c in chunks})
    lines=[f"✅ Indexed **{len(paths)} file(s)**, **{sum(x['pages'] for x in summary)} pages**, **{len(chunks)} chunks** in **{elapsed:.1f}s**."]
    for x in summary: lines.append(f"- {x['file']}: {x['pages']} pages; OCR pages: {x['ocr_pages'] or 'none'}")
    lines.append("- Document types: "+", ".join(types))
    return "\n".join(lines),[asdict(c) for c in chunks],gr.update(choices=["All"]+types,value="All")

def chat_ui(message,history,k,doc_type,auto_route):
    history=history or []
    if ENGINE.index is None:
        reply="Please upload and process documents first."
    elif not message.strip(): reply="Please enter a question."
    else:
        result=ENGINE.answer(message,int(k),doc_type,bool(auto_route))
        src="\n".join(f"- {s['source']}, page {s['page']} | {s['document_type']} | score {s['score']:.3f}" for s in result['sources'])
        reply=(f"{result['answer']}\n\n**Confidence:** {result['confidence']:.1%}  |  "
               f"**Chunks used:** {result['chunks_used']}\n\n**Sources**\n{src or '- None'}")
    return history+[{"role":"user","content":message},{"role":"assistant","content":reply}],""

CSS=""".gradio-container{max-width:1350px!important;margin:auto!important}.hero{background:linear-gradient(100deg,#132a4f,#0f7c82);color:white;padding:18px;border-radius:16px;text-align:center}.metric{border-left:5px solid #0f7c82;padding-left:12px}"""
with gr.Blocks(title="Full RAG Studio") as demo:
    gr.Markdown("# Full RAG Studio\nOCR-enabled, metadata-aware document intelligence with citations",elem_classes="hero")
    chunk_state=gr.State([])
    with gr.Row():
        with gr.Column(scale=1):
            files=gr.File(label="Upload digital or scanned PDFs",file_types=[".pdf"],file_count="multiple")
            process=gr.Button("Process and Index",variant="primary")
            info=gr.Markdown("No documents indexed.",elem_classes="metric")
            gr.Markdown("### Retrieval settings")
            doc_type=gr.Dropdown(["All"],value="All",label="Document type filter")
            auto_route=gr.Checkbox(True,label="Auto-route by question")
            k=gr.Slider(1,8,value=4,step=1,label="Chunks to retrieve")
            clear=gr.Button("Clear chat")
        with gr.Column(scale=2):
            chat=gr.Chatbot(label="Grounded answers",height=520)
            question=gr.Textbox(label="Question",placeholder="Ask a question about the indexed documents...",lines=2)
            send=gr.Button("Ask",variant="primary")
    process.click(process_ui,files,[info,chunk_state,doc_type],show_progress="full")
    send.click(chat_ui,[question,chat,k,doc_type,auto_route],[chat,question],show_progress="full")
    question.submit(chat_ui,[question,chat,k,doc_type,auto_route],[chat,question],show_progress="full")
    clear.click(lambda:([],""),outputs=[chat,question])
if __name__=="__main__": demo.launch(share=True,css=CSS)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6597766618ff10b319.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Test checklist
1. Upload a digital PDF and a scanned PDF.
2. Confirm page, OCR, chunk and document-type information.
3. Ask a specific question and inspect citations, confidence and chunk count.
4. Change document filter, auto-routing and top-k.
5. Test an unsupported question to confirm the grounded fallback.